# Curs 1 — Primul apel către un LLM
## LLM-uri, API-uri și parametri de generare
În acest notebook facem primul apel către un model Gemini folosind Python.
Scopul nu este să construim încă un agent, ci să înțelegem structura minimă a unui apel către un LLM:
- modelul folosit
- mesajele trimise către model
- parametrii de generare
- outputul primit
La finalul notebook-ului trebuie să avem:
1. API key configurată corect în `.env`
2. un prim apel funcțional către Gemini
3. un rezumat neutru generat de model
4. un mic experiment cu `temperature`
5. un output salvat împreună cu parametrii folosiți
---


## 0. Instalare pachete

In [1]:
# Rulează o singură dată. Dacă ești în Google Colab, sari peste acest pas.
%pip install -q openai python-dotenv requests


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Instalare biblioteci
Folosim biblioteca `openai`, dar nu folosim un model OpenAI.
Folosim Gemini prin endpoint-ul compatibil cu formatul OpenAI.
-  sintaxa `messages = [{"role": "...", "content": "..."}]` este foarte răspândită și ne va ajuta mai târziu să lucrăm și cu OpenRouter, DeepSeek sau alte modele compatibile.

## 2. Configurarea cheii API

https://aistudio.google.com/api-keys

Cheia API nu se scrie direct în notebook.
O salvăm într-un fișier local numit `.env`.
Fișierul `.env` trebuie să conțină:
```text
GEMINI_API_KEY=cheia_ta_aici

!Important:

nu urca niciodată .env pe GitHub
în repository trebuie să existe doar .env.example
dacă ai publicat cheia accidental, șterge cheia din Google AI Studio și generează alta

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from google import genai


In [2]:
import sys
!{sys.executable} -m pip install google-genai


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Creăm clientul API
Clientul este obiectul prin care trimitem cereri către model.
Aici avem trei lucruri importante:
- `api_key`: cheia personală
- `base_url`: adresa endpoint-ului Gemini compatibil cu formatul OpenAI
- `model`: modelul pe care îl vom folosi în apel

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Missing GEMINI_API_KEY. Add it to your .env file.")

In [4]:
from openai import OpenAI

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL = "gemini-2.5-flash"

print("Client configurat.")
print("Model selectat:", MODEL)

Client configurat.
Model selectat: gemini-2.5-flash


## 4. Primul apel minimal
Începem cu cel mai simplu caz: trimitem o singură întrebare către model și primim un răspuns.
Aici nu folosim încă `system message`, nu definim un rol special și nu construim o conversație.
Vrem doar să verificăm că:
- cheia API funcționează
- modelul răspunde
- înțelegem forma minimă a unui apel




In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say hello."}
    ]
)
print(response.choices[0].message.content)

Hello!


In [6]:
# raspunsul modelului 
response.choices[0].message.content

'Hello!'

In [7]:
response

ChatCompletion(id='1ozzaZnSCOjxnsEPiPnk6Qs', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1777568982, model='gemini-2.5-flash', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=4, total_tokens=29, completion_tokens_details=None, prompt_tokens_details=None))

- ChatCompletion(...) = obiectul complet returnat de API. Nu este doar textul, ci răspunsul + metadata.
- id='JRvvaaXoMueynsEPusrdsQM' = identificator unic al apelului. Util pentru debugging/loguri.
- object='chat.completion' = tipul obiectului returnat. Confirmă că este un răspuns de tip chat completion.
- created=1777277734 = timestamp intern al momentului când a fost creat răspunsul.
- model='gemini-2.5-flash' = modelul care a generat răspunsul.
- choices=[...] = lista de răspunsuri generate. De obicei folosim primul răspuns: choices[0].
- index=0 = poziția răspunsului în lista choices. Aici este primul și singurul răspuns.
- message=ChatCompletionMessage(...) = mesajul generat de model.
- message.content='Hello!' = textul efectiv al răspunsului. Îl accesezi cu:
- response.choices[0].message.content
- message.role='assistant' = rolul mesajului generat. Modelul răspunde ca assistant.
- finish_reason='stop' = modelul s-a oprit normal. Nu a fost tăiat de limită.
- logprobs=None = nu ai cerut probabilități pentru tokeni.
- refusal=None = modelul nu a refuzat cererea.
- annotations=None = nu există adnotări suplimentare returnate.
- audio=None = nu există output audio.
- function_call=None = modelul nu a cerut apelarea unei funcții vechi-style.
- tool_calls=None = modelul nu a cerut folosirea unui tool.
- service_tier=None = nu apare un nivel special de serviciu raportat.
- system_fingerprint=None = providerul nu a returnat o amprentă internă a sistemului.
- usage=CompletionUsage(...) = informații despre tokeni consumați.
- prompt_tokens=4 = tokenii trimiși către model în cerere.
- completion_tokens=2 = tokenii generați de model în răspuns.
- total_tokens=31 = totalul raportat de provider. La Gemini prin compatibilitate OpenAI poate include overhead intern, deci nu trebuie tratat mereu ca simpla sumă prompt + completion.
- completion_tokens_details=None și prompt_tokens_details=None = nu există detalii suplimentare despre tokeni.

In [8]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": ""
            "You are a clear teacher explaining AI to sociology students."
        },
        {
            "role": "user",
            "content": "Explain what a large language model is in 3 simple sentences."
        }
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

A Large Language Model (LLM) is an advanced artificial intelligence trained on an enormous amount of text and code from the internet. Through this extensive training, it learns to understand patterns, grammar, facts, and even the nuances of human language. This enables it to generate coherent and contextually relevant text, answer questions, summarize information, or even write creative content, much like a human would.


## 5. Mai multe modele

In [9]:
models_to_test = [
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
]

for model_name in models_to_test:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": "Explain what an LLM is in one simple sentence."}
        ]
    )
    print("=" * 60)
    print("MODEL:", model_name)
    print(response.choices[0].message.content)

MODEL: gemini-2.5-flash
An LLM is an AI model trained on a massive amount of text data to understand, generate, and respond in human-like language.
MODEL: gemini-2.5-flash-lite
An LLM is a type of AI that can understand and generate human-like text by learning from massive amounts of data.


In [10]:
response

ChatCompletion(id='64zzaevlL-jsnsEPz9ryCQ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='An LLM is a type of AI that can understand and generate human-like text by learning from massive amounts of data.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1777569003, model='gemini-2.5-flash-lite', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=25, prompt_tokens=12, total_tokens=37, completion_tokens_details=None, prompt_tokens_details=None))

## 6. Exercițiu: rezumat neutru al unei știri
Vom da modelului un text scurt și îi cerem să producă un rezumat neutru.
Regulă:
- maximum 60 de cuvinte
- fără opinie personală
- fără exagerări
- fără informații care nu apar în text

In [11]:
news_text = """
Guvernul a anunțat un nou program de finanțare pentru modernizarea transportului public în orașele mari.
Programul include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital.
Reprezentanții ministerului spun că scopul este reducerea poluării și creșterea accesului la transport public.
Unele organizații civice au cerut criterii clare de alocare a fondurilor și transparență în implementare.
"""

prompt = f"""
Rezuma textul de mai jos în maximum 60 de cuvinte.
Rezumatul trebuie să fie neutru, clar și factual.
Nu adăuga informații care nu apar în text.

TEXT:
{news_text}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You summarize public-interest texts in a neutral and factual way."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2,
    
)

summary = response.choices[0].message.content

print(summary)

Guvernul a anunțat un program de finanțare pentru modernizarea transportului public în orașele mari. Acesta include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital. Ministerul afirmă că scopul este reducerea poluării și creșterea accesului. Organizațiile civice au cerut criterii clare de alocare a fondurilor și transparență în implementare.


## 7. Ce face `temperature`?
`temperature` controlează cât de variat este răspunsul.
- valori mici: răspunsuri mai stabile, mai previzibile
- valori mari: răspunsuri mai creative, dar uneori mai puțin precise
Pentru analiză socială, clasificare și rezumate factuale, începem de obicei cu valori mici: `0`, `0.2`, `0.3`.
Pentru generare creativă putem folosi valori mai mari.

In [12]:
test_prompt = """
Descrie în două propoziții ce este un model lingvistic mare.
Explică pentru studenți de sociologie, fără jargon tehnic.
"""

temperatures = [0.0, 0.7, 1.5]

for temp in temperatures:
    response = client.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {
                "role": "system",
                "content": "You explain AI concepts clearly and briefly."
            },
            {
                "role": "user",
                "content": test_prompt
            }
        ],
        temperature=temp,
        max_tokens=120
    )

    print("=" * 80)
    print("TEMPERATURE:", temp)
    print(response.choices[0].message.content)

TEMPERATURE: 0.0
Un model lingvistic mare este ca un asistent super inteligent care a citit o cantitate uriașă de texte, de la cărți la articole de pe internet. Datorită acestei "lecturi" extinse, el poate înțelege și genera limbaj uman, ajutându-ne să scriem, să rezumăm informații sau chiar să purtăm conversații.
TEMPERATURE: 0.7
Un model lingvistic mare este un program de calculator antrenat pe o cantitate uriașă de text, cum ar fi cărți și articole de pe internet, pentru a înțelege și genera limbaj uman. Acest lucru îi permite să răspundă la întrebări, să scrie texte creative sau să rezume informații, ca și cum ar fi o persoană care a citit foarte mult.
TEMPERATURE: 1.5
Imaginează-ți un model lingvistic mare ca pe un fel de "super-lector" care a citit o cantitate uriașă de texte, de la cărți la articole pe internet. Pe baza a tot ce a "citit", el poate să înțeleagă și să creeze texte similare cu cele umane, fiind capabil să răspundă la întrebări sau să scrie povești.


## 8. Mini-interpretare
Completați după rulare:
1. Care răspuns a fost cel mai stabil?
2. Care răspuns a fost cel mai creativ?
3. Care răspuns este mai potrivit pentru cercetare academică?
4. Ce valoare de `temperature` ați folosi pentru rezumate neutre?
TODO:
Scrieți aici observațiile voastre.

## 9. Funcție simplă pentru apeluri repetate


In [13]:
MODEL = "gemini-2.5-flash-lite"
def ask_model(
    user_message,
    system_message="You are a helpful assistant.",
    model=MODEL,
    temperature=0.3,
    max_tokens=300
):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response.choices[0].message.content

In [14]:
answer = ask_model(
    user_message="Explain the difference between an LLM and an AI agent in 4 bullet points.",
    system_message="You explain AI concepts to sociology students using simple language.",
    temperature=0.3,
    max_tokens=250
)

print(answer)

Here's the difference between an LLM and an AI agent, explained simply for sociology students:

*   **LLM (Large Language Model): The Super-Smart Talker.** Think of an LLM like a really, really well-read person who can understand and generate human-like text. It's great at answering questions, writing stories, summarizing things, and having conversations, but it mostly just *reacts* to what you tell it.

*   **AI Agent: The Goal-Oriented Doer.** An AI agent is like a person who not only can talk but also has a specific job or goal to achieve. It uses its "brain" (which might include an LLM) to *plan* actions, *make decisions*, and *take steps* to get things done in the real or digital world.

*   **LLM is a Tool, Agent is the Worker.** An LLM is a powerful tool that an AI agent can *use*. The agent is the one who decides *how* to use that tool (and other tools) to accomplish its mission. The LLM is the engine, but the agent is the driver with a destination.

*   **Action vs.


## 10. Salvăm outputul și parametrii


Trebuie să știm:
- ce model am folosit
- ce prompt am trimis
- ce parametri am setat
- ce răspuns am primit


In [15]:
import json
from datetime import datetime
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

experiment = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL,
    "temperature": 0.2,
    "max_tokens": 120,
    "task": "neutral_news_summary",
    "input_text": news_text,
    "output_text": summary
}

output_path = output_dir / "c1_first_summary.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(experiment, f, ensure_ascii=False, indent=2)

print("Experiment salvat în:", output_path)

Experiment salvat în: outputs\c1_first_summary.json


## 11. Verificăm fișierul salvat
Acum citim fișierul înapoi, ca să vedem că rezultatul a fost salvat corect.

In [16]:
with open(output_path, "r", encoding="utf-8") as f:
    saved_experiment = json.load(f)

saved_experiment

{'timestamp': '2026-04-30T20:11:48.515064',
 'model': 'gemini-2.5-flash-lite',
 'temperature': 0.2,
 'max_tokens': 120,
 'task': 'neutral_news_summary',
 'input_text': '\nGuvernul a anunțat un nou program de finanțare pentru modernizarea transportului public în orașele mari.\nProgramul include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital.\nReprezentanții ministerului spun că scopul este reducerea poluării și creșterea accesului la transport public.\nUnele organizații civice au cerut criterii clare de alocare a fondurilor și transparență în implementare.\n',
 'output_text': 'Guvernul a anunțat un program de finanțare pentru modernizarea transportului public în orașele mari. Acesta include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital. Ministerul afirmă că scopul este reducerea poluării și creșterea accesului. Organizațiile civice au cerut criterii clare de alocare a fondurilo

## 12. Exercițiu scurt pentru studenți
Alegeți un text scurt, de 5–8 rânduri, dintr-o sursă publică.
Poate fi despre transport, sănătate, educație, mediu sau costul vieții.
Completați variabila de mai jos și generați un rezumat neutru.
Apoi salvați rezultatul.

In [17]:
student_text = """
TODO: Lipiți aici un text scurt dintr-o sursă publică.
"""

student_prompt = f"""
Rezuma textul de mai jos în maximum 60 de cuvinte.
Rezumatul trebuie să fie neutru, clar și factual.
Nu adăuga informații care nu apar în text.

TEXT:
{student_text}
"""

student_summary = ask_model(
    user_message=student_prompt,
    system_message="You summarize public-interest texts in a neutral and factual way.",
    temperature=0.2,
    max_tokens=120
)

print(student_summary)

Vă rog să furnizați textul pe care doriți să îl rezum. Voi crea un rezumat neutru, clar și factual, de maximum 60 de cuvinte, bazat exclusiv pe informațiile din textul dumneavoastră.


In [18]:
student_experiment = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL,
    "temperature": 0.2,
    "max_tokens": 120,
    "task": "student_neutral_summary",
    "input_text": student_text,
    "output_text": student_summary
}

student_output_path = output_dir / "c1_student_summary.json"

with open(student_output_path, "w", encoding="utf-8") as f:
    json.dump(student_experiment, f, ensure_ascii=False, indent=2)

print("Experiment salvat în:", student_output_path)

Experiment salvat în: outputs\c1_student_summary.json


## 13. Checklist pentru finalul Cursului 1
La finalul acestui notebook trebuie să aveți:
- API key funcțională
- `.env` local configurat
- primul apel către Gemini rulat cu succes
- un rezumat neutru generat
- experimentul cu `temperature` rulat
- output salvat în `outputs/`
- `.env` adăugat în `.gitignore`
Pentru repository:
- `README.md`
- `.env.example`
- `.gitignore`
- notebook-ul de Curs 1
- fără cheia API în GitHub

## 14. Ce luăm mai departe în Cursul 2
Acum știm să apelăm un model și să controlăm minim răspunsul.
În Cursul 2 vom compara modele:
- același prompt
- modele diferite
- criterii simple de evaluare: claritate, factualitate, neutralitate, viteză
Întrebarea următoare nu mai este „cum chemăm modelul?”, ci „ce model alegem pentru proiect?”.